# Semana 4: Regresión Logística y Métricas de Clasificación
## Notebook de Ejercicios (NB2) – Detección de Spam

**Propósito:** Aplicar regresión logística a un problema real de clasificación de textos (detección de spam) y evaluar su rendimiento con las métricas adecuadas.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Convertir texto a características numéricas usando CountVectorizer y TF-IDF.
- Entrenar una regresión logística y evaluar con precisión, recall y F1.
- Analizar la matriz de confusión y decidir la métrica más relevante para el negocio.
- Experimentar con diferentes valores de C (regularización) y umbrales de decisión.
- Aplicar SMOTE para manejar el desbalance de clases y observar su impacto.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline

# Para SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Carga del Dataset: SMS Spam Collection

Cargamos el dataset directamente desde la URL pública del UCI Machine Learning Repository. Este dataset contiene 5,574 mensajes SMS etiquetados como 'ham' (legítimo) o 'spam'.

In [ ]:
# URL del dataset (UCI Machine Learning Repository)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'

# Descargamos y leemos
import zipfile
import requests
from io import BytesIO

response = requests.get(url)
with zipfile.ZipFile(BytesIO(response.content)) as z:
    with z.open('SMSSpamCollection') as f:
        df = pd.read_csv(f, sep='\t', header=None, names=['label', 'message'])

print(f"Dimensiones del dataset: {df.shape}")
print("\nPrimeras 5 filas:")
df.head()

In [ ]:
# Distribución de clases
print("Distribución de etiquetas:")
print(df['label'].value_counts())

# Porcentajes
print("\nPorcentajes:")
print(df['label'].value_counts(normalize=True) * 100)

# Visualización
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='label')
plt.title('Distribución de Mensajes')
plt.show()

### 1.1. Breve exploración de los mensajes

Veamos algunos ejemplos de cada clase para entender la naturaleza de los textos.

In [ ]:
# Mostrar algunos mensajes de cada clase
print("=== Ejemplos de HAM (legítimos) ===")
for msg in df[df['label'] == 'ham']['message'].head(3):
    print(f"- {msg}")

print("\n=== Ejemplos de SPAM ===")
for msg in df[df['label'] == 'spam']['message'].head(3):
    print(f"- {msg}")

---
## 2. Preparación de Datos

Convertimos las etiquetas a formato binario (spam=1, ham=0) y dividimos en entrenamiento y prueba.

In [ ]:
# Convertir etiquetas a binario
df['label_bin'] = (df['label'] == 'spam').astype(int)

# Verificar
df[['label', 'label_bin']].head(10)

In [ ]:
# Dividir en entrenamiento y prueba (estratificado para mantener proporciones)
X_train, X_test, y_train, y_test = train_test_split(
    df['message'], 
    df['label_bin'], 
    test_size=0.2, 
    random_state=42,
    stratify=df['label_bin']
)

print(f"Tamaño de entrenamiento: {len(X_train)}")
print(f"Tamaño de prueba: {len(X_test)}")
print(f"\nProporción de spam en entrenamiento: {y_train.mean():.2%}")
print(f"Proporción de spam en prueba: {y_test.mean():.2%}")

---
## 3. Vectorización de Texto

Convertimos los mensajes de texto en vectores numéricos usando dos enfoques:
1. **CountVectorizer**: frecuencia de cada palabra.
2. **TfidfVectorizer**: frecuencia ponderada por inversa de documento (TF-IDF).

Probaremos ambos y compararemos.

In [ ]:
# CountVectorizer
count_vect = CountVectorizer(stop_words='english', max_features=5000)
X_train_count = count_vect.fit_transform(X_train)
X_test_count = count_vect.transform(X_test)

print(f"CountVectorizer - Dimensiones entrenamiento: {X_train_count.shape}")
print(f"Número de características: {len(count_vect.get_feature_names_out())}")

In [ ]:
# TfidfVectorizer
tfidf_vect = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)

print(f"TfidfVectorizer - Dimensiones entrenamiento: {X_train_tfidf.shape}")
print(f"Número de características: {len(tfidf_vect.get_feature_names_out())}")

---
## 4. Entrenamiento y Evaluación con Regresión Logística

Entrenamos un modelo con cada vectorizador y comparamos resultados.

In [ ]:
# Modelo con CountVectorizer
lr_count = LogisticRegression(C=1, max_iter=1000, random_state=42)
lr_count.fit(X_train_count, y_train)
y_pred_count = lr_count.predict(X_test_count)
y_proba_count = lr_count.predict_proba(X_test_count)[:, 1]

# Métricas
print("=== Regresión Logística con CountVectorizer ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_count):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_count):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_count):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_count):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba_count):.4f}")

print("\nMatriz de confusión:")
cm_count = confusion_matrix(y_test, y_pred_count)
print(cm_count)

In [ ]:
# Modelo con TfidfVectorizer
lr_tfidf = LogisticRegression(C=1, max_iter=1000, random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = lr_tfidf.predict(X_test_tfidf)
y_proba_tfidf = lr_tfidf.predict_proba(X_test_tfidf)[:, 1]

print("=== Regresión Logística con TfidfVectorizer ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_tfidf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_tfidf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_tfidf):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_tfidf):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba_tfidf):.4f}")

print("\nMatriz de confusión:")
cm_tfidf = confusion_matrix(y_test, y_pred_tfidf)
print(cm_tfidf)

### 4.1. Visualización de matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm_count, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('CountVectorizer')
axes[0].set_xlabel('Predicho')
axes[0].set_ylabel('Real')

sns.heatmap(cm_tfidf, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('TfidfVectorizer')
axes[1].set_xlabel('Predicho')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

### 4.2. Curvas ROC comparativas

In [ ]:
fpr_count, tpr_count, _ = roc_curve(y_test, y_proba_count)
fpr_tfidf, tpr_tfidf, _ = roc_curve(y_test, y_proba_tfidf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_count, tpr_count, 'b-', label=f'Count (AUC = {roc_auc_score(y_test, y_proba_count):.3f})')
plt.plot(fpr_tfidf, tpr_tfidf, 'r-', label=f'TF-IDF (AUC = {roc_auc_score(y_test, y_proba_tfidf):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curvas ROC - Comparación')
plt.legend()
plt.grid(True)
plt.show()

---
## 5. Análisis para el Negocio: ¿Qué métrica es más importante?

En un sistema de detección de spam, debemos considerar los costos de cada tipo de error:

- **Falso Positivo (FP)**: Un mensaje legítimo (ham) es marcado como spam y va a la carpeta de spam. El usuario podría perder un mensaje importante.
- **Falso Negativo (FN)**: Un mensaje spam llega a la bandeja de entrada. El usuario recibe publicidad no deseada o, peor, podría ser víctima de phishing.

**¿Qué es peor?** Depende del contexto:
- Si priorizamos **no perder mensajes importantes**, debemos minimizar los FP (alta precisión).
- Si priorizamos **proteger al usuario de spam y phishing**, debemos minimizar los FN (alto recall).

En la práctica, muchos sistemas optan por un equilibrio, pero suelen dar más peso a evitar FN (recall alto) porque el daño potencial de un spam malicioso es mayor que el de un falso positivo (que el usuario puede revisar en la carpeta de spam).

In [ ]:
# Usamos el mejor modelo (por ejemplo, TF-IDF) para análisis
print("=== Análisis de errores ===")
tn, fp, fn, tp = cm_tfidf.ravel()
print(f"Verdaderos Negativos (ham bien clasificado): {tn}")
print(f"Falsos Positivos (ham marcado como spam): {fp}")
print(f"Falsos Negativos (spam no detectado): {fn}")
print(f"Verdaderos Positivos (spam bien clasificado): {tp}")

print(f"\nTasa de Falsos Positivos: {fp/(tn+fp):.2%}")
print(f"Tasa de Falsos Negativos: {fn/(tp+fn):.2%}")

# Mostramos algunos ejemplos de errores
errores = X_test[(y_test != y_pred_tfidf).values]
errores_reales = y_test[(y_test != y_pred_tfidf).values]
errores_pred = y_pred_tfidf[(y_test != y_pred_tfidf).values]

print("\n=== Ejemplos de errores ===")
for i in range(min(5, len(errores))):
    print(f"Real: {'spam' if errores_reales.iloc[i] else 'ham'} | Pred: {'spam' if errores_pred[i] else 'ham'}")
    print(f"Mensaje: {errores.iloc[i][:100]}...")
    print()

---
## 6. Experimentación con Diferentes Valores de C (Regularización)

Probamos diferentes valores de C (inverso de la regularización) y observamos el efecto en las métricas.

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
resultados_C = []

for C in C_values:
    lr = LogisticRegression(C=C, max_iter=1000, random_state=42)
    lr.fit(X_train_tfidf, y_train)
    y_pred = lr.predict(X_test_tfidf)
    
    resultados_C.append({
        'C': C,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred)
    })

df_C = pd.DataFrame(resultados_C)
print("=== Efecto de C en las métricas ===")
df_C.round(4)

In [ ]:
# Visualización
plt.figure(figsize=(10, 6))
plt.plot(df_C['C'], df_C['Precision'], 'o-', label='Precision')
plt.plot(df_C['C'], df_C['Recall'], 's-', label='Recall')
plt.plot(df_C['C'], df_C['F1'], '^-', label='F1')
plt.xscale('log')
plt.xlabel('C (escala log)')
plt.ylabel('Métrica')
plt.title('Efecto de la regularización en las métricas')
plt.legend()
plt.grid(True)
plt.show()

---
## 7. Experimentación con Umbrales de Decisión

El umbral por defecto es 0.5. Podemos ajustarlo para favorecer recall (detectar más spam) a costa de precisión, o viceversa.

In [ ]:
# Usamos el mejor modelo (por ejemplo, C=1 con TF-IDF)
lr_best = LogisticRegression(C=1, max_iter=1000, random_state=42)
lr_best.fit(X_train_tfidf, y_train)
y_proba = lr_best.predict_proba(X_test_tfidf)[:, 1]

umbrales = np.arange(0.1, 0.9, 0.05)
resultados_umbral = []

for umbral in umbrales:
    y_pred_umbral = (y_proba >= umbral).astype(int)
    resultados_umbral.append({
        'Umbral': umbral,
        'Accuracy': accuracy_score(y_test, y_pred_umbral),
        'Precision': precision_score(y_test, y_pred_umbral, zero_division=0),
        'Recall': recall_score(y_test, y_pred_umbral, zero_division=0),
        'F1': f1_score(y_test, y_pred_umbral, zero_division=0)
    })

df_umbral = pd.DataFrame(resultados_umbral)
print("=== Efecto del Umbral de Decisión ===")
df_umbral.round(4)

In [ ]:
# Visualización
plt.figure(figsize=(10, 6))
plt.plot(df_umbral['Umbral'], df_umbral['Precision'], 'o-', label='Precision')
plt.plot(df_umbral['Umbral'], df_umbral['Recall'], 's-', label='Recall')
plt.plot(df_umbral['Umbral'], df_umbral['F1'], '^-', label='F1')
plt.xlabel('Umbral')
plt.ylabel('Métrica')
plt.title('Efecto del umbral de decisión en las métricas')
plt.legend()
plt.grid(True)
plt.show()

---
## 8. (Opcional) Aplicación de SMOTE

Aplicamos SMOTE para generar muestras sintéticas de la clase minoritaria (spam) y ver si mejora la detección.

In [ ]:
# Creamos un pipeline con SMOTE y regresión logística
smote_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('lr', LogisticRegression(C=1, max_iter=1000, random_state=42))
])

# Entrenamos (nota: SMOTE solo se aplica al entrenamiento)
smote_pipeline.fit(X_train_tfidf, y_train)
y_pred_smote = smote_pipeline.predict(X_test_tfidf)

print("=== Modelo con SMOTE ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_smote):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_smote):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_smote):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_smote):.4f}")

print("\nMatriz de confusión con SMOTE:")
cm_smote = confusion_matrix(y_test, y_pred_smote)
print(cm_smote)

In [ ]:
# Comparación final
comparacion = pd.DataFrame({
    'Modelo': ['CountVectorizer', 'TF-IDF', 'TF-IDF + SMOTE'],
    'Accuracy': [accuracy_score(y_test, y_pred_count), accuracy_score(y_test, y_pred_tfidf), accuracy_score(y_test, y_pred_smote)],
    'Precision': [precision_score(y_test, y_pred_count), precision_score(y_test, y_pred_tfidf), precision_score(y_test, y_pred_smote)],
    'Recall': [recall_score(y_test, y_pred_count), recall_score(y_test, y_pred_tfidf), recall_score(y_test, y_pred_smote)],
    'F1': [f1_score(y_test, y_pred_count), f1_score(y_test, y_pred_tfidf), f1_score(y_test, y_pred_smote)]
})

print("=== Comparación Final ===")
comparacion.round(4)

---
## 9. Conclusiones

Hemos aplicado regresión logística a un problema real de detección de spam:

✔️ **Vectorización**: CountVectorizer y TF-IDF convierten texto en características numéricas.
✔️ **Evaluación**: Precisión, recall y F1 son más informativos que la exactitud en problemas desbalanceados.
✔️ **Matriz de confusión**: Permite analizar los tipos de error y su costo para el negocio.
✔️ **Regularización (C)**: Valores muy pequeños o muy grandes pueden degradar el rendimiento.
✔️ **Umbral de decisión**: Ajustarlo permite priorizar recall o precisión según necesidades.
✔️ **SMOTE**: Puede mejorar el recall, aunque a veces reduce la precisión.

**Recomendación para el negocio:** Dado que el costo de un falso negativo (spam no detectado) puede ser alto (phishing, estafas), priorizaríamos un modelo con alto recall, incluso sacrificando algo de precisión. En este caso, el modelo con SMOTE ofrece un buen balance.

---
**Fin del Notebook de Ejercicios - Semana 4**